In [1]:
from Bio import SeqIO
import pandas as pd
from src import database
from Bio.SeqFeature import SeqFeature, FeatureLocation, AfterPosition

In [2]:
accessions = pd.read_csv('data/cox2_th_test_sim/round2/primer-trim-new/new_acc.txt', header=None)[0]
accessions

0    JF793557.1
1    KC145166.1
2    MF838769.1
Name: 0, dtype: str

In [3]:
gb_records = [record for record in SeqIO.parse('data/cox2_th_test_sim/round2/input-sequences.gb', format='genbank') if record.id in accessions.values]
gb_records

[SeqRecord(seq=Seq('TCCTGCAACTCCTGTTATGGAAGGTATTATAAATTTTCATCATGATTTAATGTA...TAT'), id='JF793557.1', name='JF793557', description='Pustula sp. MT-2011 cytochrome oxidase subunit 2 gene, partial cds; mitochondrial', dbxrefs=[]),
 SeqRecord(seq=Seq('ATGGAAGGTATTATAAACTTTCACCATGATTTAATGTTTTTCTTAATTATTGTA...TGG'), id='KC145166.1', name='KC145166', description='Pythium phragmiticola isolate P56 cytochrome oxidase subunit II (cox II) gene, partial cds; mitochondrial', dbxrefs=[]),
 SeqRecord(seq=Seq('ATTATAAATTTTCATAACGATTTAATGTTTTTTTTAATAGTTATTGTTATTTTT...GAA'), id='MF838769.1', name='MF838769', description='Sirolpidium heterosiphoniae isolate KNU-M-omc4 cytochrome oxidase subunit II gene, partial cds; mitochondrial', dbxrefs=[])]

In [4]:
hmm_hits = database.parse_hmm_tblout('data/cox2_th_test_sim/round2/hmm/sequence-scores.tsv')
hmm_hits.dtypes

target_name             str
target_accession        str
query_name              str
query_accession         str
hmm_from              int64
hmm_to                int64
ali_from              int64
ali_to                int64
env_from              int64
env_to                int64
sq_len                int64
strand                  str
e_value             float64
score               float64
bias                float64
description             str
dtype: object

In [ ]:
keywords = set()

def is_hit_in_feature(hit: pd.Series, feature: SeqFeature):
    feat_start = int(feature.location.start)
    feat_end = int(feature.location.end)
    return feat_start <= hit['ali_from'] <= feat_end or feat_start <= hit['ali_to'] <= feat_end

for record in gb_records:
    for feature in record.features:
        # Is feature a gene?
        if feature.type != 'CDS':
            continue
        # Find HMM hit location
        hit: pd.Series = hmm_hits.loc[hmm_hits['target_name'] == record.id].iloc[0]
        print(hit)
        if is_hit_in_feature(hit, feature):
            # Add feature name to keyword list
            keywords.update(feature.qualifiers['product'])
keywords

target_name         JF793557.1
target_accession             -
query_name              db-msa
query_accession              -
hmm_from                     1
hmm_to                     288
ali_from                     5
ali_to                     293
env_from                     5
env_to                     293
sq_len                     561
strand                       +
e_value                    0.0
score                    240.6
bias                      61.3
description                  -
Name: 41, dtype: object
target_name         KC145166.1
target_accession             -
query_name              db-msa
query_accession              -
hmm_from                    12
hmm_to                     288
ali_from                     1
ali_to                     277
env_from                     1
env_to                     277
sq_len                     563
strand                       +
e_value                    0.0
score                    266.1
bias                      48.8
description    

{'cytochrome oxidase subunit 2', 'cytochrome oxidase subunit II'}